# Pacemaker Current Simulations

This notebook explores the effects of pacemaker currents on dopaminergic neuron dynamics, including the corrected model with physiological activation kinetics.

---

# *This file allows to generate all plots necessary for figures*

# **Useful packages and functions**

## Dependencies and Setup

In [ ]:
using DifferentialEquations, Plots, Polynomials, LaTeXStrings, ColorSchemes, DelimitedFiles, DataFrames
using Statistics, StatsPlots, Random, ProgressMeter, Printf, LinearAlgebra, Plots.PlotMeasures
include("DA_kinetics.jl") # Loading of DA kinetics of gating variables
include("DA_models.jl") # Loading of DA model
include("DA_utils.jl"); # Loading of some utils functions

# **Global variables**

## Model Parameters

In [ ]:
# Definition of simulation time (in ms)
const Tfinal = 20000
const tspan  = (0.0, Tfinal)
tt = 0. : 0.01 : Tfinal
tt_rand = 0. : 1 : Tfinal

# Definition of reversal potential values (in mV), [Mg] and membrane capacitance
const VNa     = 60. # Sodium reversal potential
const VK      = -90. # Potassium reversal potential
const VCa     = 50. # Calcium reversal potential
const VH      = -29. # H reversal potential
const VLNS    = -65. # Leak reversal potential
const EPacemaker = 4.2732015978991615 # Reversal potential of pacemaking channels

const C       = 1. # Membrane capacitance
const fCa     = 0.018 # Fraction of unbuffered free calcium
const ICapmax = 11 # Maximum calcium pump current
const F       = 96520 # Faraday constant in ms*µA/mmol (and taking cm³=mL)
const d       = 15 # Soma diameter in cm
const L       = 25 # Soma length

# Definition of voltage range for the DICs
const Vmin = -100 
const Vmax = 50
const Vrange = range(Vmin, stop=Vmax, step=0.0154640);

## Plotting Configuration

In [ ]:
# Modifying backend GR attributes
gr(guidefontsize=25, tickfontsize=15, legendfontsize=12, margin=5Plots.mm, grid=false)
myApple = RGBA(187/255, 206/255, 131/255, 1)
mySalmon = RGBA(243/255, 124/255, 130/255)
myYellow = RGBA(228/255, 205/255, 121/255, 1)
myBlue = RGBA(131/255, 174/255, 218/255, 1)
myDarkBlue = RGBA(114/255, 119/255, 217/255, 1)
myOrange = RGBA(241/255, 175/255, 113/255, 1)
myPink = RGBA(243/255, 124/255, 130/255, 1)
myPurple = RGBA(169/255, 90/255, 179/255, 1)
myGreen = RGBA(132/255, 195/255, 168/255, 1)
myRed = RGBA(158/255, 3/255, 8/255, 1)
myGray = RGBA(150/255, 150/255, 150/255, 1)
myLightBlue = RGBA(127/255, 154/255, 209/255, 1);
default(fmt = :png);

In [ ]:
# Define a struct (optional, but useful if you need parameters)
struct NoisyFunction
    amplitude::Float64  # amplitude of the noise
end

# Overload the () operator to make the struct callable
function (nf::NoisyFunction)(x::Float64)
    noise = nf.amplitude * randn()  # Generate Gaussian noise (mean 0, std 1)
    return noise  # Example function with noise
end

function condition(u,t,integrator) # Event when event_f(u,t) == 0
  (u[1]- (-20.))
end

function affect!(integrator)
end

cb = ContinuousCallback(condition, affect!, nothing, save_positions = (true, false));

In [ ]:
"""
    compute_charge_transfer(sol, p, t_start, t_end)

Compute the charge transfer (in Coulombs) for each ionic current over a time interval.

# Arguments
- `sol`: ODE solution object from DifferentialEquations.jl
- `p`: Parameter tuple (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)
- `t_start`: Start time of integration period (in ms)
- `t_end`: End time of integration period (in ms)

# Returns
A named tuple with charge transfer for each current in Coulombs (C)
"""
function compute_charge_transfer(sol, p, t_start, t_end)
    
    # Create dense time array for integration
    tt = range(t_start, t_end, length=10000)
    
    # Extract solution at these time points
    x = sol(tt)
    dt = tt[2] - tt[1]  # Time step in ms
    
    # Extract state variables
    V = x[1, :]
    m = x[2, :]
    h = x[3, :]
    hs = x[4, :]
    l = x[5, :]
    n = x[6, :]
    p_var = x[7, :]
    q1 = x[8, :]
    q2 = x[9, :]
    o = x[10, :]
    i_var = x[11, :]
    mH = x[12, :]
    Ca = x[13, :]
    
    # Extract conductances from parameters
    gNa = p[2]
    gCaL = p[3]
    gKd = p[4]
    gKA = p[5]
    gKERG = p[6]
    gKSK = p[7]
    gH = p[8]
    gLNS = p[9]
    gLCa = p[10]
    gPacemaker = p[11]
    
    # Compute SK_inf for each voltage point
    SK_inf = zeros(length(V))
    for idx in 1:length(V)
        if Ca[idx] > 0
            SK_inf[idx] = 1/(1+(0.00019/Ca[idx])^4)
        end
    end
    
    # Compute pacemaker activation
    m_Pace = mPacemaker_inf.(V)
    
    # Compute currents in pA (convert from nS * mV to pA)
    # Current = conductance * gating * (V - E_rev)
    # Multiply by (pi*d*L/100) to get actual current in pA
    scale_factor = pi * d * L / 100
    
    I_Na = scale_factor .* gNa .* m.^3 .* h .* hs .* (V .- VNa)
    I_CaL = scale_factor .* gCaL .* l .* (V .- VCa)
    I_Kd = scale_factor .* gKd .* n.^3 .* (V .- VK)
    I_KA = scale_factor .* gKA .* p_var .* (q1./2 .+ q2./2) .* (V .- VK)
    I_KERG = scale_factor .* gKERG .* o .* (V .- VK)
    I_KSK = scale_factor .* gKSK .* (V .- VK) .* SK_inf
    I_H = scale_factor .* gH .* mH.^2 .* (V .- VH)
    I_LNS = scale_factor .* gLNS .* (V .- VLNS)
    I_LCa = scale_factor .* gLCa .* (V .- VCa)
    I_Pacemaker = scale_factor .* gPacemaker .* m_Pace .* (V .- EPacemaker)
    
    # Integrate currents using trapezoidal rule
    # Q = ∫I dt, where I is in pA and t is in ms
    # Convert: pA * ms = 10^-12 A * 10^-3 s = 10^-15 C (femtocoulombs)
    # To get Coulombs, multiply by 10^-15
    conversion = 1e-15  # Convert pA*ms to Coulombs
    
    Q_Na = sum(I_Na[1:end-1] .+ I_Na[2:end]) * dt/2 * conversion
    Q_CaL = sum(I_CaL[1:end-1] .+ I_CaL[2:end]) * dt/2 * conversion
    Q_Kd = sum(I_Kd[1:end-1] .+ I_Kd[2:end]) * dt/2 * conversion
    Q_KA = sum(I_KA[1:end-1] .+ I_KA[2:end]) * dt/2 * conversion
    Q_KERG = sum(I_KERG[1:end-1] .+ I_KERG[2:end]) * dt/2 * conversion
    Q_KSK = sum(I_KSK[1:end-1] .+ I_KSK[2:end]) * dt/2 * conversion
    Q_H = sum(I_H[1:end-1] .+ I_H[2:end]) * dt/2 * conversion
    Q_LNS = sum(I_LNS[1:end-1] .+ I_LNS[2:end]) * dt/2 * conversion
    Q_LCa = sum(I_LCa[1:end-1] .+ I_LCa[2:end]) * dt/2 * conversion
    Q_Pacemaker = sum(I_Pacemaker[1:end-1] .+ I_Pacemaker[2:end]) * dt/2 * conversion
    
    # Return as named tuple
    return (
        Q_Na = Q_Na,
        Q_CaL = Q_CaL,
        Q_Kd = Q_Kd,
        Q_KA = Q_KA,
        Q_KERG = Q_KERG,
        Q_KSK = Q_KSK,
        Q_H = Q_H,
        Q_LNS = Q_LNS,
        Q_LCa = Q_LCa,
        Q_Pacemaker = Q_Pacemaker,
        Q_total = Q_Na + Q_CaL + Q_Kd + Q_KA + Q_KERG + Q_KSK + Q_H + Q_LNS + Q_LCa + Q_Pacemaker
    )
end

"""
    compute_charge_transfer_ISI(sol, p, t_spike1, t_spike2; spike_window=5.0)

Compute the charge transfer (in Coulombs) for each ionic current during the 
interspike interval, excluding the spikes themselves.

# Arguments
- `sol`: ODE solution object from DifferentialEquations.jl
- `p`: Parameter tuple (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)
- `t_spike1`: Time of first spike (in ms)
- `t_spike2`: Time of second spike (in ms)
- `spike_window`: Time window to exclude around each spike (default 5 ms total, ±2.5 ms)

# Returns
A named tuple with charge transfer for each current in Coulombs (C) during ISI only
"""
function compute_charge_transfer_ISI(sol, p, t_spike1, t_spike2; spike_window=5.0)
    # Define ISI window (exclude spike_window/2 around each spike)
    t_start = t_spike1 + spike_window/2
    t_end = t_spike2 - spike_window/2
    
    # Check if there's any ISI left after excluding spikes
    if t_end <= t_start
        @warn "ISI window too small after excluding spikes. Consider reducing spike_window."
        # Return zeros
        return (
            Q_Na = 0.0, Q_CaL = 0.0, Q_Kd = 0.0, Q_KA = 0.0, Q_KERG = 0.0,
            Q_KSK = 0.0, Q_H = 0.0, Q_LNS = 0.0, Q_LCa = 0.0, Q_Pacemaker = 0.0,
            Q_total = 0.0
        )
    end
    
    # Use the main function for the ISI portion
    return compute_charge_transfer(sol, p, t_start, t_end)
end


"""
    display_charge_transfer(charges; period=nothing)

Pretty print the charge transfer results.

# Arguments
- `charges`: Named tuple returned by compute_charge_transfer
- `period`: Optional period duration in ms for displaying transfer per second
"""
function display_charge_transfer(charges; period=nothing)
    println("\n=== Charge Transfer During Period ===")
    if !isnothing(period)
        println("Period duration: $(round(period, digits=2)) ms")
    end
    println("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    
    # Inward currents (depolarizing)
    println("\nInward Currents (Depolarizing):")
    println("  Na⁺     : $(round(charges.Q_Na * 1e15, digits=3)) fC  ($(round(charges.Q_Na * 1e12, sigdigits=3)) pC)")
    println("  CaL     : $(round(charges.Q_CaL * 1e15, digits=3)) fC  ($(round(charges.Q_CaL * 1e12, sigdigits=3)) pC)")
    println("  H       : $(round(charges.Q_H * 1e15, digits=3)) fC  ($(round(charges.Q_H * 1e12, sigdigits=3)) pC)")
    println("  Pacer   : $(round(charges.Q_Pacemaker * 1e15, digits=3)) fC  ($(round(charges.Q_Pacemaker * 1e12, sigdigits=3)) pC)")
    
    # Outward currents (repolarizing)
    println("\nOutward Currents (Repolarizing):")
    println("  Kd      : $(round(charges.Q_Kd * 1e15, digits=3)) fC  ($(round(charges.Q_Kd * 1e12, sigdigits=3)) pC)")
    println("  KA      : $(round(charges.Q_KA * 1e15, digits=3)) fC  ($(round(charges.Q_KA * 1e12, sigdigits=3)) pC)")
    println("  KERG    : $(round(charges.Q_KERG * 1e15, digits=3)) fC  ($(round(charges.Q_KERG * 1e12, sigdigits=3)) pC)")
    println("  KSK     : $(round(charges.Q_KSK * 1e15, digits=3)) fC  ($(round(charges.Q_KSK * 1e12, sigdigits=3)) pC)")
    
    # Leak currents
    println("\nLeak Currents:")
    println("  LNS     : $(round(charges.Q_LNS * 1e15, digits=3)) fC  ($(round(charges.Q_LNS * 1e12, sigdigits=3)) pC)")
    println("  LCa     : $(round(charges.Q_LCa * 1e15, digits=3)) fC  ($(round(charges.Q_LCa * 1e12, sigdigits=3)) pC)")
    
    println("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    println("  Total   : $(round(charges.Q_total * 1e15, digits=3)) fC  ($(round(charges.Q_total * 1e12, sigdigits=3)) pC)")
    
    # If period provided, show charge per second
    if !isnothing(period)
        freq = 1000.0 / period  # Convert ms to Hz
        println("\nFiring frequency: $(round(freq, digits=2)) Hz")
        println("\nCharge transfer per second:")
        for (name, value) in pairs(charges)
            if name != :Q_total
                charge_per_sec = value * freq
                println("  $(name): $(round(charge_per_sec * 1e12, sigdigits=3)) pC/s")
            end
        end
        println("  Total: $(round(charges.Q_total * freq * 1e12, sigdigits=3)) pC/s")
    end
    
    println("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")
end

# Simu true model no pacemaker

In [ ]:
# gNa        = 25. # Sodium current maximal conductance
# gCaL       = 0#0.139 # L-type calcium current maximal conductance
# gKd        = 3. # Delayed-rectifier potassium current maximal conductance
# gKA        = 1.68 # A-type potassium current maximal conductance
# gKERG      = 0.13 # ERG current maximal conductance
# gKSK       = 0.07 # SK current maximal conductance
# gH         = 0.078 # H current maximal conductance
# gLNS       = 0.028 # Leak non specific current maximal conductance
# gLCa       = 0.00245 # Leak calcium current maximal conductance
# gPacemaker = 0#10 # Pacemaker current maximal conductance

# # Input current definition
# Iapp(t) = 0 # pA
# p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

# # Initial conditions
# V0 = 0.
# Ca0 = 1e-4
# x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
#     0., 0., mH_inf(V0), Ca0]

# # Simulation
# prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
# sol = solve(prob; maxiters=1e6); # Solving the problem

In [ ]:
# # Retrieving variables
# x         = sol(tt)
# V_plot    = x[1, :]
# voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myDarkBlue, 
#     legend=false, ylims=(-100, 30), margins=20Plots.px, size=(1200, 800))
# ylabel!("V (mV)")

# current = plot(tt./1e3, Iapp.(tt), linewidth=2.5, color=myDarkBlue,
#     legend=false, margins=20Plots.px, size=(1200, 800))
# ylabel!("I (pA)")
# xlabel!("t (s)")

# l = @layout [
#     a{1.0*w, 0.70*h}
#     b{1.0*w, 0.30*h}
# ]
# VI = plot(voltage, current, layout=l, size=(1000, 1000), margins=20Plots.px, xlims=(19, 20))
# display(VI)

# Simu instaneous pacemaker

In [ ]:
# gNa        = 25. # Sodium current maximal conductance
# gCaL       = 1. # L-type calcium current maximal conductance
# gKd        = 10. # Delayed-rectifier potassium current maximal conductance
# gKA        = 1.68 # A-type potassium current maximal conductance
# gKERG      = 0.13 # ERG current maximal conductance
# gKSK       = 0.3 # SK current maximal conductance
# gH         = 0.078 # H current maximal conductance
# gLNS       = 0.01 # Leak non specific current maximal conductance
# gLCa       = 0.00245 # Leak calcium current maximal conductance
# gPacemaker = 5 # Pacemaker current maximal conductance

# # Input current definition
# Iapp(t) = 0 # pA
# p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

# # Initial conditions
# V0 = -50.
# Ca0 = 1e-4
# x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
#     0., 0., mH_inf(V0), Ca0]

# # Simulation
# prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
# sol = solve(prob; maxiters=1e6); # Solving the problem

In [ ]:
# # Retrieving variables
# x         = sol(tt)
# V_plot    = x[1, :]
# voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myDarkBlue, 
#     legend=false, ylims=(-100, 30), margins=20Plots.px, size=(1200, 800))
# ylabel!("V (mV)")

# current = plot(tt./1e3, Iapp.(tt), linewidth=2.5, color=myDarkBlue,
#     legend=false, margins=20Plots.px, size=(1200, 800))
# ylabel!("I (pA)")
# xlabel!("t (s)")

# l = @layout [
#     a{1.0*w, 0.70*h}
#     b{1.0*w, 0.30*h}
# ]
# VI = plot(voltage, current, layout=l, size=(1000, 1000), margins=20Plots.px, xlims=(19., 20.))
# display(VI)

# Simu noisy model instantaneous

In [ ]:
# gNa        = 25. # Sodium current maximal conductance
# gCaL       = 0#0.139 # L-type calcium current maximal conductance
# gKd        = 3. # Delayed-rectifier potassium current maximal conductance
# gKA        = 1.68 # A-type potassium current maximal conductance
# gKERG      = 0.13 # ERG current maximal conductance
# gKSK       = 0.07 # SK current maximal conductance
# gH         = 0#0.078 # H current maximal conductance
# gLNS       = 0.01 # Leak non specific current maximal conductance
# gLCa       = 0.00245 # Leak calcium current maximal conductance
# gPacemaker = 5 # Pacemaker current maximal conductance

# # Parameter vector for simulations
# NoiseIntensity = 20
# p = (NoisyFunction(NoiseIntensity), gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

# # Initial conditions
# V0 = 0.
# Ca0 = 1e-4
# x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
#     0., 0., mH_inf(V0), Ca0]

# # Simulation
# prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
# sol = solve(prob; maxiters=1e7); # Solving the problem

In [ ]:
# # Retrieving variables
# x         = sol(tt)
# V_plot    = x[1, :]
# voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myDarkBlue, 
#     legend=false, ylims=(-100, 30), margins=20Plots.px, size=(1200, 800))
# ylabel!("V (mV)")

# tt_reduced = range(first(tt), stop=last(tt), length=10000)
# current = plot(tt_reduced./1e3, NoisyFunction(NoiseIntensity).(tt_reduced), linewidth=2.5, 
#     color=myDarkBlue, legend=false, margins=20Plots.px, size=(1200, 800))
# ylabel!("I (pA)")
# xlabel!("t (s)")

# l = @layout [
#     a{1.0*w, 0.70*h}
#     b{1.0*w, 0.30*h}
# ]
# VI = plot(voltage, current, layout=l, size=(1000, 1000), margins=20Plots.px, xlims=(0, 20))
# display(VI)

# Simu full model

In [ ]:
# gNa        = 25. # Sodium current maximal conductance
# gCaL       = 0#0.139 # L-type calcium current maximal conductance
# gKd        = 3. # Delayed-rectifier potassium current maximal conductance
# gKA        = 1.68 # A-type potassium current maximal conductance
# gKERG      = 0.13 # ERG current maximal conductance
# gKSK       = 0.07 # SK current maximal conductance
# gH         = 0.078 # H current maximal conductance
# gLNS       = 0.01 # Leak non specific current maximal conductance
# gLCa       = 0.00245 # Leak calcium current maximal conductance
# gPacemaker = 5 # Pacemaker current maximal conductance
# tau        = 1e0 # Time constant of pacemaking current

# # Input current definition
# Iapp(t) = 0 # pA
# p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker, tau)

# # Initial conditions
# V0 = -50.
# Ca0 = 1e-4
# x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
#       0., 0., mH_inf(V0), mPacemaker_inf(V0), Ca0]

# # Simulation
# prob = ODEProblem(DA_ODE_true_notinstant, x0, tspan, p) # Describing the problem
# sol = solve(prob; maxiters=1e6); # Solving the problem

In [ ]:
# # Retrieving variables
# x         = sol(tt)
# V_plot    = x[1, :]
# voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myDarkBlue, 
#     legend=false, ylims=(-100, 30), margins=20Plots.px, size=(1200, 800))
# ylabel!("V (mV)")

# current = plot(tt./1e3, Iapp.(tt), linewidth=2.5, color=myDarkBlue,
#     legend=false, margins=20Plots.px, size=(1200, 800))
# ylabel!("I (pA)")
# xlabel!("t (s)")

# l = @layout [
#     a{1.0*w, 0.70*h}
#     b{1.0*w, 0.30*h}
# ]
# VI = plot(voltage, current, layout=l, size=(1000, 1000), margins=20Plots.px, xlims=(19, 20))
# display(VI)

# Simu transient

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0.3 # SK current maximal conductance
gH         = 0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker_t(t) = 5 * (t > Tfinal/2) # Pacemaker current maximal conductance
tau        = 1e0 # Time constant of pacemaking current

# Input current definition
Iapp(t) = 0 # pA
p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker_t, tau)

# Initial conditions
V0 = -50.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), mPacemaker_inf(V0), Ca0]

# Simulation
prob = ODEProblem(DA_ODE_true_notinstant_transient, x0, tspan, p) # Describing the problem
sol = solve(prob, Tsit5(); maxiters=1e6); # Solving the problem

In [ ]:
# Retrieving variables
x           = sol(tt)
V_plot      = x[1, :]

voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myGray, legend=false, ylims=(-100, 30), 
    margins=20Plots.px, size=(1200, 800), xlims=(9, 15), xticks=([9, 11, 13, 15], 
            ["0", "2", "4", "6", "8"]))
ylabel!("V (mV)")
xlabel!("t (s)")

display(voltage)
# savefig(voltage, "./figures/dynamic_transient_fast.pdf")
# savefig(voltage, "./figures/dynamic_transient_fast.svg")

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0.3 # SK current maximal conductance
gH         = 0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker_t(t) = 20 * (t > Tfinal/2) # Pacemaker current maximal conductance
tau        = 35.25 # Time constant of pacemaking current

# Input current definition
Iapp(t) = 0 # pA
p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker_t, tau)

# Initial conditions
V0 = -50.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), mPacemaker_inf(V0), Ca0]

# Simulation
prob = ODEProblem(DA_ODE_true_notinstant_transient, x0, tspan, p) # Describing the problem
sol = solve(prob, Tsit5(); maxiters=1e6); # Solving the problem

In [ ]:
# Retrieving variables
x           = sol(tt)
V_plot      = x[1, :]

voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myGray, legend=false, ylims=(-100, 30), 
    margins=20Plots.px, size=(1200, 800), xlims=(9, 15), xticks=([9, 11, 13, 15], 
            ["0", "2", "4", "6", "8"]))
ylabel!("V (mV)")
xlabel!("t (s)")

display(voltage)
# savefig(voltage, "./figures/dynamic_transient_slow.pdf")
# savefig(voltage, "./figures/dynamic_transient_slow.svg")

# **Figures**

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0.3 # SK current maximal conductance
gH         = 0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Input current definition
Iapp(t) = 0 # pA
p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

# Initial conditions
V0 = -50.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Simulation
prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
sol = solve(prob, Tsit5(); maxiters=1e6); # Solving the problem

In [ ]:
sol_spike_times = solve(prob, Tsit5(); callback=cb, save_everystep=false,save_start=false,save_end=false)
chosen_spike = sol_spike_times.t[50];

In [ ]:
# Retrieving variables
x         = sol(tt)
V_plot    = x[1, :]
voltage = plot(tt, V_plot, linewidth=2.5, color=myGray, legend=false, ylims=(-70, 30), 
    margins=20Plots.px, size=(1200, 800), xlims=(chosen_spike-10, chosen_spike+10), 
    xticks=([chosen_spike-10, chosen_spike-5, chosen_spike, chosen_spike+5, chosen_spike+10], 
            ["0", "5", "10", "15", "20"]))
ylabel!("V (mV)")
xlabel!("t (ms)")
display(voltage)
# savefig(voltage, "./figures/fig3_single_spike.pdf")
# savefig(voltage, "./figures/fig3_single_spike.svg")

In [ ]:
# Retrieving variables
x           = sol(tt)
V_plot      = x[1, :]
m_Pace_plot = mPacemaker_inf.(V_plot)

voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myGray, legend=false, ylims=(-100, 30), 
    margins=20Plots.px, size=(1200, 800), xlims=(19, 20), xticks=false)
ylabel!("V (mV)")

current = plot(tt./1e3, (pi*d*L/100) .* gPacemaker .* m_Pace_plot .* (V_plot .- EPacemaker), linewidth=2.5, color=myGray,
    legend=false, margins=20Plots.px, size=(1200, 800), xticks=([19, 19.5, 20], 
            ["0", "0.5", "1"]), ylims=(-10, 1))
ylabel!("I XG (pA)")
xlabel!("t (s)")

l = @layout [
    a{1.0*w, 0.70*h}
    b{1.0*w, 0.30*h}
]
VI = plot(voltage, current, layout=l, size=(1000, 1000), margins=20Plots.px, xlims=(19, 20))
display(VI)
# savefig(VI, "./figures/fig3_full_trace_all_g.pdf")
# savefig(VI, "./figures/fig3_full_trace_all_g.svg")

In [ ]:
# Get two consecutive spike times
spike_50 = sol_spike_times.t[50]
spike_51 = sol_spike_times.t[51]

# Full period (including spikes)
charges_full = compute_charge_transfer(sol, p, spike_50, spike_51)
display_charge_transfer(charges_full, period=(spike_51 - spike_50))

# ISI only (excluding spikes with 10ms window)
window=10.0
charges_ISI = compute_charge_transfer_ISI(sol, p, spike_50, spike_51, spike_window=window)
isi_duration = spike_51 - spike_50 - window
display_charge_transfer(charges_ISI, period=isi_duration)

In [ ]:
function plot_charge_comparison(charges_full, charges_ISI; unit="fC")
    # Extract current names (exclude Q_total)
    current_names = ["Na", "CaL", "Kd", "KA", "KERG", "KSK", "H", "LNS", "LCa", "Pace"]
    field_names = [:Q_Na, :Q_CaL, :Q_Kd, :Q_KA, :Q_KERG, :Q_KSK, :Q_H, :Q_LNS, :Q_LCa, :Q_Pacemaker]
    
    # Convert to appropriate unit
    if unit == "fC"
        conversion = 1e15
        unit_label = "fC"
    elseif unit == "pC"
        conversion = 1e12
        unit_label = "pC"
    else
        error("Unit must be 'fC' or 'pC'")
    end
    
    # Extract values
    full_values = [getfield(charges_full, field) * conversion for field in field_names]
    ISI_values = [getfield(charges_ISI, field) * conversion for field in field_names]
    
    # Create positions for bars
    x = 1:length(current_names)
    
    # Create the plot
    p = plot(size=(1200, 800), ylims=(-4000, 5000),
             dpi=300,
             bottom_margin=10Plots.px,
             left_margin=15Plots.px,
             right_margin=10Plots.px,
             top_margin=10Plots.px)
    
    # Plot ISI only (solid bars)
    bar!(x, ISI_values,
         label="ISI only",
         color=:steelblue,
         alpha=0.8,
         bar_width=0.35,
         xticks=(x, current_names),
         ylabel="Charge Transfer ($unit_label)",
         legend=:topleft,
         legendfontsize=12,
         tickfontsize=11,
         guidefontsize=13)
    
    # Plot full period (hatched bars) - slightly offset
    bar!(x .+ 0.35, full_values,
         label="Full Period",
         color=:coral,
         alpha=0.7,
         bar_width=0.35,
         fillstyle=:auto)
    
    # Add a horizontal line at zero for reference
    hline!([0], color=:black, linestyle=:dash, linewidth=1, label="")
    
    # Rotate x-axis labels if needed
    plot!(xrotation=45)
    
    return p
end

In [ ]:
p1= plot_charge_comparison(charges_full, charges_ISI; unit="fC")
display(p1)
# savefig(p1, "./figures/charges.pdf")
# savefig(p1, "./figures/charges.svg")

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0.3 # SK current maximal conductance
gH         = 0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Parameter vector for simulations
Iapp(t) = 0
p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

# Initial conditions
V0 = 0.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Simulation
prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
sol = solve(prob, Tsit5(); maxiters=1e6); # Solving the problem

In [ ]:
# Retrieving variables
x           = sol(tt)
V_plot      = x[1, :]
m_Pace_plot = mPacemaker_inf.(V_plot)

voltage = plot(tt./1e3, V_plot, linewidth=2.5, color=myGray, legend=false, ylims=(-100, 30), 
    margins=20Plots.px, size=(1200, 800), xlims=(19, 20), xticks=false)
ylabel!("V (mV)")

current = plot(tt./1e3, (pi*d*L/100) .* gPacemaker .* m_Pace_plot .* (V_plot .- EPacemaker), linewidth=2.5, color=myGray,
    legend=false, margins=20Plots.px, size=(1200, 800), xticks=([19, 19.5, 20], 
            ["0", "0.5", "1"]), ylims=(-10, 1))
ylabel!("I XG (pA)")
xlabel!("t (s)")

l = @layout [
    a{1.0*w, 0.70*h}
    b{1.0*w, 0.30*h}
]
VI = plot(voltage, current, layout=l, size=(1000, 1000), margins=20Plots.px, xlims=(19., 19.7))
display(VI)
# savefig(VI, "./figures/fig3_full_trace_Na_Kd.pdf")
# savefig(VI, "./figures/fig3_full_trace_Na_Kd.svg")

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0.3 # SK current maximal conductance
gH         = 0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_XG = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

    # Simulation
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 20
        freqs_XG[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_XG[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_our_model.dat", freqs_XG);

In [ ]:
Iapps = range(0, 20, length=100)
freqs_XG = readdlm("./data/freqs_FI_our_model.dat")
FI = plot(Iapps, freqs_XG, seriestype=:scatter, legend=:topleft, linewidth=2.5, color=myGray, 
          size=(1200, 800), markersize=10, label=false, legendfontsize=20, margins=20Plots.px)
ylabel!("f (Hz)")
xlabel!("Applied current (pA)")
ylims!((0, 10))
display(FI)
# savefig(FI, "./figures/fig3_FIs_XG.pdf")
# savefig(FI, "./figures/fig3_FIs_XG.svg")

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 0#1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 0#1.68 # A-type potassium current maximal conductance
gKERG      = 0#0.13 # ERG current maximal conductance
gKSK       = 0#0.3 # SK current maximal conductance
gH         = 0#0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0#0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_XG = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

    # Simulation
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 20
        freqs_XG[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_XG[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_our_model_NaKd.dat", freqs_XG);

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0#0.3 # SK current maximal conductance
gH         = 0#0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_XG = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

    # Simulation
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 20
        freqs_XG[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_XG[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_our_model_A.dat", freqs_XG);

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 0#1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0.3 # SK current maximal conductance
gH         = 0#0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_XG = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

    # Simulation
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 20
        freqs_XG[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_XG[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_our_model_SK.dat", freqs_XG);

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 0#1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0#0.3 # SK current maximal conductance
gH         = 0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_XG = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

    # Simulation
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 20
        freqs_XG[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_XG[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_our_model_H.dat", freqs_XG);

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0#0.3 # SK current maximal conductance
gH         = 0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_XG = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

    # Simulation
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 20
        freqs_XG[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_XG[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_our_model_HA.dat", freqs_XG);

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 0#1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0.3 # SK current maximal conductance
gH         = 0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_XG = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

    # Simulation
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 20
        freqs_XG[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_XG[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_our_model_HSK.dat", freqs_XG);

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0.3 # SK current maximal conductance
gH         = 0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapps = range(0, 20, length=100)

# Initializing variables
freqs_XG = zeros(length(Iapps))

for (i, Iapp_i) in enumerate(Iapps)
    display(i)
    Iapp(t) = Iapp_i # pA
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

    # Simulation
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
    sol = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)
    
    if length(sol.t) < 20
        freqs_XG[i] = 0
    else
        # Extracting ISIs
        spike_times = sol.t
        filter!(x -> x ≥ 5000, spike_times)
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        freqs_XG[i] = 1000/mean(ISIs)
    end
end

writedlm("./data/freqs_FI_our_model_HASK.dat", freqs_XG);

In [ ]:
Iapps = range(0, 20, length=100)
freqs_XG_NaKd = readdlm("./data/freqs_FI_our_model_NaKd.dat")
freqs_XG_H = readdlm("./data/freqs_FI_our_model_H.dat")
freqs_XG_SK = readdlm("./data/freqs_FI_our_model_SK.dat")
freqs_XG_A = readdlm("./data/freqs_FI_our_model_A.dat")
freqs_XG_HSK = readdlm("./data/freqs_FI_our_model_HSK.dat")
freqs_XG_HA = readdlm("./data/freqs_FI_our_model_HA.dat")
freqs_XG_HASK = readdlm("./data/freqs_FI_our_model_HASK.dat")
FI = plot(Iapps, freqs_XG_NaKd, seriestype=:scatter, legend=:topleft, linewidth=2.5, 
          size=(1200, 800), markersize=10, label="NaKd", legendfontsize=20, margins=20Plots.px)
plot!(Iapps, freqs_XG_H, seriestype=:scatter, linewidth=2.5, markersize=10, label="H")
plot!(Iapps, freqs_XG_SK, seriestype=:scatter, linewidth=2.5, markersize=10, label="SK")
plot!(Iapps, freqs_XG_A, seriestype=:scatter, linewidth=2.5, markersize=10, label="A")
plot!(Iapps, freqs_XG_HSK, seriestype=:scatter, linewidth=2.5, markersize=10, label="HSK")
plot!(Iapps, freqs_XG_HA, seriestype=:scatter, linewidth=2.5, markersize=10, label="HA")
plot!(Iapps, freqs_XG_HASK, seriestype=:scatter, linewidth=2.5, markersize=10, label="HASK")
ylabel!("f (Hz)")
xlabel!("Applied current (pA)")
ylims!((0, 50))
display(FI)
# savefig(FI, "./figures/FIs_comparison.pdf")
# savefig(FI, "./figures/FIs_comparison.svg")

In [ ]:
gNa        = 25. # Sodium current maximal conductance
gCaL       = 1. # L-type calcium current maximal conductance
gKd        = 10. # Delayed-rectifier potassium current maximal conductance
gKA        = 1.68 # A-type potassium current maximal conductance
gKERG      = 0.13 # ERG current maximal conductance
gKSK       = 0#0.3 # SK current maximal conductance
gH         = 0#0.078 # H current maximal conductance
gLNS       = 0.01 # Leak non specific current maximal conductance
gLCa       = 0.00245 # Leak calcium current maximal conductance
gPacemaker = 5 # Pacemaker current maximal conductance

# Initial conditions
V0 = -60.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

# Input current definition
Iapp(t) = 20 # pA
p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

# Simulation
prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
sol = solve(prob);

In [ ]:
# Retrieving variables
x           = sol(tt)
V_plot      = x[1, :]

VI = plot(tt./1e3, V_plot, linewidth=2.5, color=myGray, legend=false, ylims=(-100, 30), 
    margins=20Plots.px, size=(1200, 800), xlims=(19, 20), xticks=([19, 19.5, 20], ["0", "0.5", "1"]))
ylabel!("V (mV)")
xlabel!("t (s)")

display(VI)
# savefig(VI, "./figures/FIs_Iapp_20_A.pdf")
# savefig(VI, "./figures/FIs_Iapp_20_A.svg")